# Cost — Gradient-Boosted Tree + Gurobi ML Optimization (2026)

**Part 1** reproduces Saba's tuned Gradient Boosting model *exactly* (train R² = 0.9515, test R² = 0.8432).
**Part 2** embeds that trained tree into a Gurobi model with `gurobi_ml.add_predictor_constr` and finds the
design that **minimizes predicted Cost** (all 13 features are decision variables).

The setup cell mounts Drive and finds the data automatically. Part 1 runs anywhere; Part 2 needs a Gurobi license.

## Setup — mount Google Drive & locate the data (run once)

This mounts your Drive and finds the two data files automatically, so you never have to upload them.
If your Drive layout differs, edit `DRIVE_FOLDER` below.

In [1]:
# Mount Google Drive and locate the data files automatically
import os, glob
try:
    from google.colab import drive
    drive.mount("/content/drive")          # authorize once per session
except Exception:
    pass                                    # not on Colab -> skip mounting

TRAIN_NAME = "RET2025 -Sustainability_ Outputs_Train.xlsx"
TEST_NAME  = "Test.xlsx"

# Drive folder holding this notebook + the two .xlsx files (adjust only if your Drive differs)
DRIVE_FOLDER = ("SEAR Labs/Projects/Sponsored Project/RET Green Building/Optimization Code/"
                "Treed_Regression_Optimization/Splited code_currently_used/"
                "Part2_Optimization/GB_Gurobi_Optimization")

CANDIDATES = [
    f"/content/drive/MyDrive/{DRIVE_FOLDER}",
    f"/content/drive/Shareddrives/{DRIVE_FOLDER}",
    "/content",   # fallback: files uploaded straight into the session
    ".",          # local run
]
BASE = next((d for d in CANDIDATES
             if os.path.exists(os.path.join(d, TRAIN_NAME))
             and os.path.exists(os.path.join(d, TEST_NAME))), None)

if BASE is None:                            # last resort: search the Drive (can be slow)
    hits = glob.glob(f"/content/drive/**/{TRAIN_NAME}", recursive=True)
    if hits:
        BASE = os.path.dirname(hits[0])

assert BASE is not None, ("Could not find the data files. Set BASE manually to the folder "
                          "containing the two .xlsx files.")
TRAIN = os.path.join(BASE, TRAIN_NAME)
TEST  = os.path.join(BASE, TEST_NAME)
print("Data folder:", BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data folder: /content/drive/MyDrive/SEAR Labs/Projects/Sponsored Project/RET Green Building/Optimization Code/Treed_Regression_Optimization/Splited code_currently_used/Part2_Optimization/GB_Gurobi_Optimization


## Part 1 — Reproduce Saba's tuned Gradient Boosting model

In [2]:
import pandas as pd, numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

# The RET2025 workbooks use verbose headers; map the 18 columns BY POSITION to Saba's
# canonical names. (Cols 12-14 hold CFM25 / Duct / heat pump -> HVAC_Efficiency/Duct_R/Duct_Leakage.)
CANON = ["User Assigned","# Run","Wall_Ext_Finish","Ceiling_R","Wall_R","Window_Frame",
         "Window_Glass","Window_Gain","Window_Gas","Roof_Radiant","Roof_Ext_Finish",
         "Water_Heater","HVAC_Efficiency","Duct_R","Duct_Leakage","Cost","GWP","HHP"]
CAT = ["Wall_Ext_Finish","Window_Frame","Window_Glass","Window_Gain","Window_Gas",
       "Roof_Radiant","Roof_Ext_Finish","Water_Heater","HVAC_Efficiency","Duct_R","Duct_Leakage"]

def load(path):
    df = pd.read_excel(path, header=0)
    df.columns = CANON[:df.shape[1]]                     # positional rename
    df = df[df["Cost"].notna()].reset_index(drop=True)   # keep evaluated designs (test: 47)
    for c in ["Ceiling_R", "Wall_R", "Cost"]:
        df[c] = df[c].astype(float)
    for c in CAT:
        df[c] = df[c].astype(str).str.strip()            # trim stray whitespace (e.g. 'None ')
    df["Roof_Radiant"] = df["Roof_Radiant"].replace("nan", "None")
    return df

train = load(TRAIN); test = load(TEST)

# ---- feature specification (exactly Saba's) ----------------------------------
ORDERED = {                                   # ordinal: physical low -> high order
    "Window_Gain":     ["Low", "High"],
    "HVAC_Efficiency": ["8 CFM25", "4 CFM25"],
    "Duct_R":          ["Uninsulated", "R-4", "R-6", "R-8"],
    "Duct_Leakage":    ["SEER2 14.3, 7.5 HSPF2", "SEER2 17.1, 8.2 HSPF2", "SEER2 20.9, 8.9 HSPF2"],
}
UNORDERED = ["Wall_Ext_Finish","Window_Frame","Window_Glass","Window_Gas",
             "Roof_Radiant","Roof_Ext_Finish","Water_Heater"]
NUMERIC   = ["Ceiling_R", "Wall_R"]
DROP      = ["User Assigned", "# Run", "Cost", "GWP", "HHP"]

def encode(df):
    df = df.drop(columns=[c for c in DROP if c in df.columns]).copy()
    for c, order in ORDERED.items():
        df[c] = df[c].map({lv: i for i, lv in enumerate(order)})  # ordinal integers
    return pd.get_dummies(df, columns=UNORDERED, drop_first=True)  # 2-level -> single dummy

X_train = encode(train); y_train = train["Cost"]
X_test  = encode(test);  y_test  = test["Cost"]
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
X_train = X_train.astype(float); X_test = X_test.astype(float)   # clean float for gurobi_ml
FEATURES = list(X_train.columns)

gb = GradientBoostingRegressor(learning_rate=0.05, max_depth=1, min_samples_leaf=1,
                               n_estimators=200, subsample=0.7, random_state=42)
gb.fit(X_train, y_train)

for nm, X, y in [("TRAIN", X_train, y_train), ("TEST", X_test, y_test)]:
    p = gb.predict(X)
    print(f"{nm}: R2 = {r2_score(y, p):.10f}   RMSE = {np.sqrt(mean_squared_error(y, p)):.4f}")
print("\nExpected (Saba): Train R2 = 0.9514985669 | Test R2 = 0.8431517134")
print("\nEncoded feature order (this is the order Gurobi must use):")
print(FEATURES)


TRAIN: R2 = 0.9514985669   RMSE = 46.1968
TEST: R2 = 0.8431517134   RMSE = 74.9828

Expected (Saba): Train R2 = 0.9514985669 | Test R2 = 0.8431517134

Encoded feature order (this is the order Gurobi must use):
['Ceiling_R', 'Wall_R', 'Window_Gain', 'HVAC_Efficiency', 'Duct_R', 'Duct_Leakage', 'Wall_Ext_Finish_Light', 'Window_Frame_Non-Metal', 'Window_Glass_Low E Triple', 'Window_Gas_Argon', 'Roof_Radiant_None', 'Roof_Ext_Finish_Asphalt Shingles, Light', 'Water_Heater_Tankless']


## Part 2 — Embed the tree in Gurobi and minimize Cost

Decision variables for the 13 features:

| features | Gurobi variable |
|---|---|
| 7 two-level categoricals (one-hot dummies) | binary |
| 4 ordinals (`Window_Gain`, `HVAC_Efficiency` → [0,1]; `Duct_R` → [0,3]; `Duct_Leakage` → [0,2]) | bounded integer |
| `Ceiling_R` ∈ [0, 60], `Wall_R` ∈ [7, 19] | **continuous**, bounded by data min/max |

It is a pure **MILP** — no `NonConvex`. All categoricals here are 2-level, so **no `sum=1` constraints** are needed.

In [3]:
%pip install -q gurobipy gurobi-machinelearning

If you use a **Gurobi WLS license**, configure it before building the model (fill in your credentials):

```python
import gurobipy as gp
env = gp.Env(params={"WLSACCESSID": "...", "WLSSECRET": "...", "LICENSEID": 000000})
# then create the model with:  m = gp.Model("GB_Cost_min", env=env)
```

In [4]:
import gurobipy as gp
from gurobipy import GRB
from gurobi_ml import add_predictor_constr

m = gp.Model("GB_Cost_min")          # add env=env here if using a WLS license
xvars = {}                            # feature name -> gurobipy var (the 1-row input)

# (1) numeric design vars = CONTINUOUS, bounded by the training-data min/max
for v in NUMERIC:
    lb, ub = float(X_train[v].min()), float(X_train[v].max())
    xvars[v] = m.addVar(lb=lb, ub=ub, vtype=GRB.CONTINUOUS, name=v)
    print(f"  {v}: continuous in [{lb:g}, {ub:g}]")

# (2) ordinal design vars = bounded integers, physical order preserved
for v, order in ORDERED.items():
    xvars[v] = m.addVar(vtype=GRB.INTEGER, lb=0, ub=len(order) - 1, name=v)

# (3) one-hot dummies (all 2-level) = free binaries
for c in [c for c in FEATURES if c not in NUMERIC and c not in ORDERED]:
    safe = c.replace(" ", "_").replace(",", "")           # Gurobi-safe var name
    xvars[c] = m.addVar(vtype=GRB.BINARY, name=safe)

m.update()

# assemble the 1-row input in the EXACT training feature order, then embed the tree
input_df = pd.DataFrame([{c: xvars[c] for c in FEATURES}])[FEATURES]
cost = m.addVar(lb=-GRB.INFINITY, name="Cost_pred")
pred_constr = add_predictor_constr(m, gb, input_df, cost, epsilon=1e-5)
pred_constr.print_stats()

m.setObjective(cost, GRB.MINIMIZE)
m.optimize()

err = np.max(np.abs(np.asarray(pred_constr.get_error()).ravel()))
print(f"\nEmbedding check  max|Gurobi - sklearn| = {err:.3e}   (should be ~0)")
print(f"Optimal predicted Cost = {cost.X:.4f}")


Restricted license - for non-production use only - expires 2027-11-29
  Ceiling_R: continuous in [0, 60]
  Wall_R: continuous in [7, 19]
Model for gbtree_reg:
600 variables
601 constraints
800 general constraints
Input has shape (1, 13)
Output has shape (1, 1)

--------------------------------------------------------------------------------
Estimator       Output Shape    Variables              Constraints              
                                                Linear    Quadratic      General
tree                  (1, 1)            2            3            0            4

tree0                 (1, 1)            2            3            0            4

tree1                 (1, 1)            2            3            0            4

tree2                 (1, 1)            2            3            0            4

tree3                 (1, 1)            2            3            0            4

tree4                 (1, 1)            2            3            0            4

tre

In [ ]:
# ---- decode the optimal design back to human-readable levels -----------------
def ref_other(v):
    levs = sorted(set(train[v].fillna("None").astype(str)))
    return levs[0], levs[1]           # (reference = all-zero, other = dummy==1)

# raw decode from the Gurobi solution (with epsilon, continuous numerics sit just past a split)
raw = {}
for v in NUMERIC:                 raw[v] = float(xvars[v].X)
for v, order in ORDERED.items():  raw[v] = order[int(round(xvars[v].X))]
for v in UNORDERED:
    ref, oth = ref_other(v)
    raw[v] = oth if round(xvars[f"{v}_{oth}"].X) == 1 else ref

DESIGN_COLS = NUMERIC + list(ORDERED) + UNORDERED
def _pred(dd):   # GB prediction for a human-readable design
    row = pd.concat([train[DESIGN_COLS], pd.DataFrame([{c: dd[c] for c in DESIGN_COLS}])], ignore_index=True)
    return float(gb.predict(encode(row).reindex(columns=FEATURES, fill_value=0).astype(float).iloc[[-1]])[0])

# Snap each continuous numeric to the BUILDABLE catalog value that keeps the SAME predicted
# cost. The tree is piecewise-constant, so within a split cell the value can move to a real
# R-value without changing any leaf -> the design becomes buildable and matches the discrete file.
CATALOG = {v: sorted(map(float, pd.unique(train[v]))) for v in NUMERIC}
target  = float(cost.X)
design  = dict(raw)
for v in NUMERIC:
    same = [val for val in CATALOG[v] if abs(_pred({**design, v: val}) - target) < 1e-6]
    design[v] = int(min(same, key=lambda x: abs(x - raw[v]))) if same else int(round(raw[v]))

print("=== Gurobi-optimal design (snapped to buildable catalog values) ===")
for k in DESIGN_COLS:
    print(f"  {k:18s}: {design[k]}")
print(f"\n  Predicted Cost (Gurobi)             : {cost.X:.2f}")
print(f"  Predicted Cost (sklearn, buildable) : {_pred(design):.2f}")
moved = {v: (round(raw[v], 3), design[v]) for v in NUMERIC if abs(raw[v] - design[v]) > 1e-9}
if moved:
    print("  snapped (continuous -> buildable):", {v: f"{a} -> {b}" for v, (a, b) in moved.items()})


## Part 3 — Score the candidate pool (comparison)

Runs the 167 pool designs (120 new + 47 evaluated) through the **same** GB model to find the lowest
predicted-Cost candidate, and compares it to the Gurobi global optimum. The pool uses R-canonical
column names, so we map them to Saba's names and clean identically (strip whitespace; blank
`RadiantBarrier` → `None`).

In [6]:
# ── Part 3: score the candidate pool through the SAME GB model ────────────────
POOL_NAME = "Pool-new-120+47_CLEANED.xlsx"
pool_path = next((os.path.join(d, POOL_NAME) for d in CANDIDATES
                  if os.path.exists(os.path.join(d, POOL_NAME))), None)
assert pool_path, f"{POOL_NAME} not found next to the data files."
pool = pd.read_excel(pool_path)

# map the pool's column names to Saba's, then clean exactly like train/test
RENAME = {"WallFinish":"Wall_Ext_Finish","CeilingInsulation":"Ceiling_R","WallInsulation":"Wall_R",
          "WindowFrameType":"Window_Frame","WindowGlassCategory":"Window_Glass","WindowGainType":"Window_Gain",
          "WindowGasType":"Window_Gas","RadiantBarrier":"Roof_Radiant","RoofMaterial":"Roof_Ext_Finish",
          "WaterHeaterType":"Water_Heater","CFM25":"HVAC_Efficiency","Duct":"Duct_R","AirSourceHeatPump":"Duct_Leakage"}
pool = pool.rename(columns=RENAME)
for c in NUMERIC: pool[c] = pool[c].astype(float)
for c in (UNORDERED + list(ORDERED)): pool[c] = pool[c].astype(str).str.strip()
pool["Roof_Radiant"] = pool["Roof_Radiant"].replace("nan", "None")   # blank radiant barrier = None

# encode with the SAME pipeline and align to the trained feature order
Xpool = encode(pool).reindex(columns=FEATURES, fill_value=0).astype(float)
pool["pred_Cost"] = gb.predict(Xpool)

ranked = pool.sort_values("pred_Cost").reset_index(drop=True)
best   = ranked.iloc[0]
n_new  = int((pool["Status"] == "new_candidate").sum())
n_test = int((pool["Status"] == "already_evaluated_test").sum())
print(f"Pool: {len(pool)} designs ({n_new} new + {n_test} evaluated)\n")
print("Lowest predicted-Cost designs in the pool:")
print(ranked[["RunNumber", "Status", "pred_Cost"]].head(8).to_string(index=False))

best_new = ranked[ranked["Status"] == "new_candidate"].iloc[0]
print(f"\nBest NEW candidate : run {best_new['RunNumber']}  predicted Cost = {best_new['pred_Cost']:.2f}")
print(f"Best design overall: run {best['RunNumber']} ({best['Status']})  predicted Cost = {best['pred_Cost']:.2f}")
print("Best pool design:", {c: (round(float(best[c]), 1) if c in NUMERIC else best[c])
                            for c in (NUMERIC + list(ORDERED) + UNORDERED)})

# compare to the Gurobi global optimum (if Part 2 has been run)
try:
    gap = best["pred_Cost"] - cost.X
    print(f"\nGurobi global optimum : {cost.X:.2f}")
    print(f"Best pool candidate   : {best['pred_Cost']:.2f}   (gap {gap:+.2f})")
    print("  -> best pool design IS the global optimum." if abs(gap) < 1e-6
          else "  -> Gurobi finds a cheaper design than anything in the pool.")
except NameError:
    print("\n(Run Part 2 to compare against the Gurobi global optimum.)")


Pool: 167 designs (120 new + 47 evaluated)

Lowest predicted-Cost designs in the pool:
 RunNumber                 Status   pred_Cost
        37          new_candidate 1670.617811
       167 already_evaluated_test 1686.555578
        14          new_candidate 1688.607620
        11          new_candidate 1689.375231
        78          new_candidate 1690.365332
        48          new_candidate 1695.207952
       127 already_evaluated_test 1697.729479
       154 already_evaluated_test 1699.617650

Best NEW candidate : run 37  predicted Cost = 1670.62
Best design overall: run 37 (new_candidate)  predicted Cost = 1670.62
Best pool design: {'Ceiling_R': 19.0, 'Wall_R': 19.0, 'Window_Gain': 'High', 'HVAC_Efficiency': '4 CFM25', 'Duct_R': 'R-8', 'Duct_Leakage': 'SEER2 20.9, 8.9 HSPF2', 'Wall_Ext_Finish': 'Light', 'Window_Frame': 'Insulated', 'Window_Glass': 'Low E Triple', 'Window_Gas': 'Air', 'Roof_Radiant': 'Double-Sided, Foil', 'Roof_Ext_Finish': 'Asphalt Shingles, Dark', 'Water_Heater': 

## Part 4 — Export the winning designs

Writes Excel **and** CSV to the Drive folder with three rows: the **Gurobi global optimum** (Part 2),
the **best pool design by simple sorting** (Part 3), and the **cheapest actually-simulated design in
the test set** (run 44). Each row has its full design spec, the GB **predicted** cost, and — where it
exists — the real **actual** cost. (Note: `HVAC_Efficiency` = CFM25/duct-leakage, `Duct_Leakage` =
air-source heat pump — Saba's original labels.)

In [7]:
# ── Part 4: export Gurobi optimum + pool best + test-set cheapest ─────────────
DESIGN_COLS = NUMERIC + list(ORDERED) + UNORDERED

def _gb_predict(dd):   # GB prediction for a human-readable design dict
    row = pd.concat([train[DESIGN_COLS], pd.DataFrame([{c: dd[c] for c in DESIGN_COLS}])], ignore_index=True)
    X = encode(row).reindex(columns=FEATURES, fill_value=0).astype(float).iloc[[-1]]
    return float(gb.predict(X)[0])
def _num(v, c): return round(float(v), 3) if c in NUMERIC else v

rows = []
# (1) Gurobi global optimum (needs Part 2)
try:
    g = {"Source": "Gurobi global optimum", "RunNumber": ""}
    g.update({c: _num(design[c], c) for c in DESIGN_COLS})
    g["Predicted_Cost"] = round(float(cost.X), 2); g["Actual_Cost"] = ""
    rows.append(g)
except NameError:
    print("Note: Part 2 (Gurobi) not run in this session -- skipping the Gurobi row.")

# (2) best pool design by simple sorting of the 167 candidates
p = {"Source": "Pool best (simple sort of 167)", "RunNumber": int(best["RunNumber"])}
p.update({c: _num(best[c], c) for c in DESIGN_COLS})
p["Predicted_Cost"] = round(float(best["pred_Cost"]), 2); p["Actual_Cost"] = ""
rows.append(p)

# (3) cheapest ACTUALLY-SIMULATED design in the test set (real cost)
tr = test.loc[test["Cost"].idxmin()]
tdes = {c: (float(tr[c]) if c in NUMERIC else tr[c]) for c in DESIGN_COLS}
t = {"Source": "Test-set cheapest (actual)", "RunNumber": int(tr["# Run"])}
t.update({c: _num(tdes[c], c) for c in DESIGN_COLS})
t["Predicted_Cost"] = round(_gb_predict(tdes), 2)   # what the GB model predicts for it
t["Actual_Cost"]    = round(float(tr["Cost"]), 2)    # the real simulated cost
rows.append(t)

out = pd.DataFrame(rows, columns=["Source", "RunNumber"] + DESIGN_COLS + ["Predicted_Cost", "Actual_Cost"])
xlsx = os.path.join(BASE, "GB_best_designs_comparison.xlsx")
out.to_excel(xlsx, index=False)
out.to_csv(xlsx[:-5] + ".csv", index=False)
print("Saved to your Drive folder:")
print("  ", xlsx)
print("  ", xlsx[:-5] + ".csv")
out


Saved to your Drive folder:
   /content/drive/MyDrive/SEAR Labs/Projects/Sponsored Project/RET Green Building/Optimization Code/Treed_Regression_Optimization/Splited code_currently_used/Part2_Optimization/GB_Gurobi_Optimization/GB_best_designs_comparison.xlsx
   /content/drive/MyDrive/SEAR Labs/Projects/Sponsored Project/RET Green Building/Optimization Code/Treed_Regression_Optimization/Splited code_currently_used/Part2_Optimization/GB_Gurobi_Optimization/GB_best_designs_comparison.csv


,Source,RunNumber,Ceiling_R,Wall_R,Window_Gain,HVAC_Efficiency,Duct_R,Duct_Leakage,Wall_Ext_Finish,Window_Frame,Window_Glass,Window_Gas,Roof_Radiant,Roof_Ext_Finish,Water_Heater,Predicted_Cost,Actual_Cost
0,Gurobi global optimum,,10.0,17.0,Low,8 CFM25,R-6,"SEER2 20.9, 8.9 HSPF2",Dark,Insulated,Low E Triple,Air,"Double-Sided, Foil","Asphalt Shingles, Dark",Tank,1627.94,
1,Pool best (simple sort of 167),37,19.0,19.0,High,4 CFM25,R-8,"SEER2 20.9, 8.9 HSPF2",Light,Insulated,Low E Triple,Air,"Double-Sided, Foil","Asphalt Shingles, Dark",Tank,1670.62,
2,Test-set cheapest (actual),44,30.0,11.0,Low,8 CFM25,R-6,"SEER2 20.9, 8.9 HSPF2",Light,Insulated,Low E Double,Argon,None,"Asphalt Shingles, Light",Tank,1733.18,1599.09
